In [ ]:
# Install / Env (comment out if already present)
!pip -q install -U "transformers>=4.44.0" "accelerate>=0.33.0" "datasets>=2.20.0" \
"peft>=0.12.0" "bitsandbytes>=0.43.3" "evaluate" "scikit-learn" "sentencepiece" "huggingface_hub>=0.24.0"


import os
os.environ["HF_HOME"] = "/kaggle/working/hf_cache"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["WANDB_DISABLED"] = "true"

In [ ]:
from pathlib import Path
import random, numpy as np, torch

# ---- CONFIGS (edit here) -----------------------------------------------------
# Base & caching
BASE_MODEL_ID     = "Qwen/Qwen2.5-1.5B"   # any CausalLM with hidden_size
ATTN_IMPL         = "sdpa"                          # 'sdpa' recommended for PyTorch 2+

# Training mode: one of {"FREEZE_BASE_ONLY_CLS", "LORA_TRAIN_BASE_FREEZE_CLS", "LORA_AND_CLS"}
TRAIN_MODE = "LORA_TRAIN_BASE_FREEZE_CLS"


# Repos (set per mode)
REPO_ID_IN  = ""   # heads (+ optional LoRA adapter) to INIT FROM
REPO_ID_OUT = ""  # heads (+ optional LoRA adapter) to PUSH TO


# Data
DATA_PATH = "" 
TRAIN_FILE = "train.csv"
# VAL_FILE = "val.csv"
# TEST_FILE = "test.csv"
CSV_DELIM = ","
TEXT_COL = "headlines"
LABEL_COLS = ["category"]


# Token / outputs
HF_TOKEN = "" # your HF token here
OUTPUT_DIR = "" 


# Train hyperparams
MAX_SEQ_LEN = 512
NUM_EPOCHS = 2
LR = 2e-4
BATCH_SIZE = 8
GRAD_ACC_STEPS = 8
SAVE_STEPS = 40
LOG_STEPS = 20
SEED = 42
LOSS_TYPE = "ce" # "bce" or "ce"
DEBUG = False
DEBUG_FRAC = 0.10


# LoRA config (used in modes 2 & 3)
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


# -----------------------------------------------------------------------------


random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

In [3]:
from huggingface_hub import login
try:
    login(HF_TOKEN)
except Exception as e:
    print("HF login skipped or failed:", e)

In [ ]:
# Check GPU
import subprocess
try:
    print(subprocess.check_output(["nvidia-smi"]).decode())
except Exception as e:
    print("No nvidia-smi:", e)
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
# Tokenizer
from transformers import AutoTokenizer

TOKENIZER_ID = BASE_MODEL_ID

tokenizer = AutoTokenizer.from_pretrained(
    TOKENIZER_ID,
    trust_remote_code=True,
    token=HF_TOKEN,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

In [ ]:
from datasets import load_dataset, DatasetDict
from pathlib import Path

DATA_PATH = Path(DATA_PATH)  # ensure Path

# Load your single CSV (train only)
raw_ds = load_dataset(
    "csv",
    data_files={"train": str(DATA_PATH / "train.csv")},
)["train"]

# First split into train+test
train_test = raw_ds.train_test_split(test_size=0.2, seed=42)

# Split test further into validation + test
test_valid = train_test["test"].train_test_split(test_size=0.5, seed=42)

# Recombine into one DatasetDict and overwrite raw_ds
raw_ds = DatasetDict({
    "train": train_test["train"],
    "validation": test_valid["train"],
    "test": test_valid["test"]
})

# Sanity check
for split in raw_ds:
    cols = set(raw_ds[split].column_names)
    assert {"Id","headlines","category"}.issubset(cols), f"{split} is missing required columns. Found: {cols}"

raw_ds

In [7]:
# Label mappings for multi-task setup

def make_label_mappings_multi(ds, label_cols):
    l2i, i2l, nlabels = {}, {}, {}
    for col in label_cols:
        labels = list(ds["train"][col]) + list(ds["validation"][col])
        uniq = sorted(set(str(x).strip() for x in labels))
        l2i[col] = {lab: i for i, lab in enumerate(uniq)}
        i2l[col] = {i: lab for lab, i in l2i[col].items()}
        nlabels[col] = len(uniq)
    return l2i, i2l, nlabels

label2id_map, id2label_map, NUM_LABELS_MAP = make_label_mappings_multi(raw_ds, LABEL_COLS)

SIZES   = tuple(NUM_LABELS_MAP[c] for c in LABEL_COLS)
TOTAL   = sum(SIZES)

OFFSETS = np.cumsum([0] + list(SIZES))[:-1]
print("Tasks:", LABEL_COLS)
print("Class sizes per task:", SIZES, "| TOTAL:", TOTAL)


Tasks: ['category']
Class sizes per task: (2,) | TOTAL: 2


In [ ]:
# Preprocess → BCE-style concatenated one-hot labels (works for CE too by argmax)

def preprocess_fn_bce(batch):
    texts = [str(x) if x is not None else "" for x in batch[TEXT_COL]]
    enc = tokenizer(texts, truncation=True, max_length=MAX_SEQ_LEN, padding=False)

    labels_cat = []
    n_items = len(texts)
    for i in range(n_items):
        onehots = []
        for col in LABEL_COLS:
            size = NUM_LABELS_MAP[col]
            val  = str(batch[col][i]).strip()
            idx  = label2id_map[col][val]
            oh   = [0.0] * size
            oh[idx] = 1.0
            onehots.extend(oh)
        labels_cat.append(onehots)

    enc["labels"] = labels_cat  # shape [B, TOTAL]
    return enc

proc_train = raw_ds["train"].map(preprocess_fn_bce, batched=True, remove_columns=raw_ds["train"].column_names)
proc_val   = raw_ds["validation"].map(preprocess_fn_bce, batched=True, remove_columns=raw_ds["validation"].column_names)
proc_test  = raw_ds["test"].map(preprocess_fn_bce, batched=True, remove_columns=raw_ds["test"].column_names)

print(proc_train)
print(proc_val)
print(proc_test)


In [9]:
# Optional DEBUG shrinking
import math

def _take_frac(ds, frac, seed=SEED):
    n = len(ds)
    k = max(1, int(math.ceil(n * frac)))
    return ds.shuffle(seed=seed).select(range(k))

if DEBUG:
    global SAVE_STEPS, LOG_STEPS
    SAVE_STEPS = max(10, SAVE_STEPS // 10)
    LOG_STEPS  = max(1, LOG_STEPS // 4)
    proc_train = _take_frac(proc_train, DEBUG_FRAC, seed=SEED)
    proc_val   = _take_frac(proc_val,   DEBUG_FRAC, seed=SEED)
    proc_test  = _take_frac(proc_test,  DEBUG_FRAC, seed=SEED)
    print(f"[DEBUG] SAVE_STEPS={SAVE_STEPS} | LOG_STEPS={LOG_STEPS}")
    print(f"[DEBUG] Sizes — train: {len(proc_train)} | val: {len(proc_val)} | test: {len(proc_test)}")

In [ ]:
# Modeling utilities
import torch, torch.nn as nn
from transformers import AutoModelForCausalLM
from peft import PeftModel, LoraConfig, get_peft_model
import json
from huggingface_hub import HfApi, hf_hub_download

compute_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16

class MeanPooler(nn.Module):
    def forward(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).type_as(last_hidden_state)
        summed = (last_hidden_state * mask).sum(dim=1)
        denom = mask.sum(dim=1).clamp(min=1e-6)
        return summed / denom

class CausalLMClassifierModel(nn.Module):
    """Base/LoRA + classification heads. Supports BCE/CE.
    Save/load heads independently from adapter; optionally push/pull to Hub.
    """
    def __init__(self, base_with_or_without_lora, sizes, task_names, loss_type="bce"):
        super().__init__()
        self.base = base_with_or_without_lora
        self.pool = MeanPooler()
        self.sizes = list(sizes)
        self.task_names = list(task_names)
        self.loss_type = loss_type.lower()

        # hidden size (model-dependent field)
        H = getattr(getattr(self.base, "config", object()), "hidden_size", None) or \
            getattr(getattr(self.base, "config", object()), "n_embd", None)
        assert H is not None, "Hidden size not found on base model config."

        self.heads = nn.ModuleList([nn.Linear(H, c) for c in self.sizes])

        if self.loss_type == "bce":
            self.loss_fct = nn.BCEWithLogitsLoss(reduction="mean")
        elif self.loss_type == "ce":
            self.loss_fct = nn.CrossEntropyLoss()
        else:
            raise ValueError(f"Unknown loss_type: {loss_type}")

    # -------------------- forward --------------------
    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        kwargs.pop("labels", None)
        kwargs.pop("return_dict", None)
        out = self.base(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
            output_hidden_states=True,
            return_dict=True,
            **kwargs,
        )
        hidden = out.hidden_states[-1]          # [B, T, H]
        pooled = self.pool(hidden, attention_mask)  # [B, H]
        logits_list = [head(pooled) for head in self.heads]
        logits = torch.cat(logits_list, dim=1)  # [B, TOTAL]

        loss = None
        if labels is not None:
            if self.loss_type == "bce":
                labels = labels.to(logits.dtype)
                start = 0; loss_sum = 0.0
                for c, logit in zip(self.sizes, logits_list):
                    y = labels[:, start:start + c]
                    loss_sum = loss_sum + self.loss_fct(logit, y)
                    start += c
                loss = loss_sum / len(self.sizes)
            elif self.loss_type == "ce":
                start = 0; loss_sum = 0.0
                for c, logit in zip(self.sizes, logits_list):
                    y = labels[:, start:start + c].argmax(-1)
                    loss_sum = loss_sum + self.loss_fct(logit, y)
                    start += c
                loss = loss_sum / len(self.sizes)
        return {"loss": loss, "logits": logits}

    # -------------------- Heads I/O --------------------
    def save_heads_local(self, out_dir):
        out_dir = Path(out_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        torch.save({"heads": [h.state_dict() for h in self.heads]}, out_dir / "cls_heads.pt")
        with open(out_dir / "cls_config.json", "w") as f:
            json.dump({"sizes": self.sizes, "tasks": self.task_names, "loss_type": self.loss_type}, f)

    def push_heads_to_hub(self, repo_id, hf_token=None):
        tmp = Path(OUTPUT_DIR) / "_heads_tmp"
        self.save_heads_local(tmp)
        api = HfApi()
        api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=hf_token)
        api.upload_file(path_or_fileobj=str(tmp / "cls_heads.pt"), path_in_repo="cls_heads.pt", repo_id=repo_id, repo_type="model", token=hf_token)
        api.upload_file(path_or_fileobj=str(tmp / "cls_config.json"), path_in_repo="cls_config.json", repo_id=repo_id, repo_type="model", token=hf_token)
        print(f"✅ Pushed heads to {repo_id}")

    @staticmethod
    def load_heads_from_hub_into(model, repo_id, hf_token=None, device="cpu"):
        cfg_path = hf_hub_download(repo_id, "cls_config.json", token=hf_token)
        with open(cfg_path, "r") as f: cfg = json.load(f)
        assert cfg["sizes"] == model.sizes and cfg["tasks"] == model.task_names, \
            "Loaded heads config does not match model sizes/tasks."
        state = torch.load(hf_hub_download(repo_id, "cls_heads.pt", token=hf_token), map_location=device)
        for h, s in zip(model.heads, state["heads"]):
            h.load_state_dict(s)
        print(f"✅ Loaded heads from {repo_id}")

    # -------------------- Adapter I/O --------------------
    def save_adapter_local(self, out_dir):
        out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
        # Works when base is a PEFT model
        if isinstance(self.base, PeftModel):
            self.base.save_pretrained(str(out_dir))
        else:
            raise RuntimeError("Base does not have a PEFT adapter to save.")

    def push_adapter_to_hub(self, repo_id, hf_token=None):
        if not isinstance(self.base, PeftModel):
            raise RuntimeError("Base does not have a PEFT adapter to push.")
        self.base.push_to_hub(repo_id, token=hf_token)
        print(f"✅ Pushed adapter to {repo_id}")


In [11]:
# Build base model (with optional existing adapter)

def build_base_model():
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        trust_remote_code=True,
        token=HF_TOKEN,
        torch_dtype=compute_dtype,
        device_map="auto",
        attn_implementation=ATTN_IMPL,
        low_cpu_mem_usage=True,
    )
    # training niceties
    if tokenizer.pad_token is not None:
        base.config.pad_token_id = tokenizer.pad_token_id
    base.config.use_cache = False
    if hasattr(base, "gradient_checkpointing_enable"):
        base.gradient_checkpointing_enable()
    return base

In [12]:
# Mode wiring: LoRA attach / freeze switches
from peft import PeftModel as _PeftModel

def attach_lora(base):
    # If an existing adapter is provided, load it; else create fresh
    if ADAPTER_REPO_IN:
        print(f"🔗 Loading existing LoRA adapter: {ADAPTER_REPO_IN}")
        base = _PeftModel.from_pretrained(base, ADAPTER_REPO_IN, token=HF_TOKEN, is_trainable=True)
    else:
        print("✨ Creating a fresh LoRA adapter")
        lcfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
                          task_type="CAUSAL_LM", target_modules=LORA_TARGETS)
        base = get_peft_model(base, lcfg)
    return base

In [ ]:
# %% [INIT / BUILD MODEL FROM REPO_ID_IN]
from huggingface_hub import HfApi, hf_hub_download, list_repo_files
from peft import PeftModel as _PeftModel, LoraConfig, get_peft_model
import json, os
from pathlib import Path

api = HfApi()

def _repo_files(repo_id):
    try:
        return set(list_repo_files(repo_id=repo_id, repo_type="model", token=HF_TOKEN))
    except Exception:
        return set()

def _repo_has_heads(files):
    return {"cls_heads.pt", "cls_config.json"}.issubset(files)

def _repo_has_adapter(files):
    # common PEFT files
    return any(
        f in files
        for f in ["adapter_model.safetensors", "adapter_model.bin", "adapter_config.json"]
    )

def _load_heads_from_repo_into(model, repo_id, device="cpu"):
    cfg_f = hf_hub_download(repo_id, "cls_config.json", token=HF_TOKEN)
    with open(cfg_f, "r") as f:
        cfg = json.load(f)
    assert cfg["sizes"] == model.sizes and cfg["tasks"] == model.task_names, \
        "Heads config in repo doesn't match current model sizes/tasks."
    state_f = hf_hub_download(repo_id, "cls_heads.pt", token=HF_TOKEN)
    state = torch.load(state_f, map_location=device)
    for h, s in zip(model.heads, state["heads"]):
        h.load_state_dict(s)
    print(f"✅ Heads loaded from {repo_id}")

def _maybe_attach_lora(base, files):
    if _repo_has_adapter(files):
        print(f"🔗 Loading LoRA adapter from {REPO_ID_IN}")
        return _PeftModel.from_pretrained(base, REPO_ID_IN, token=HF_TOKEN, is_trainable=True)
    else:
        print("✨ Creating a fresh LoRA adapter")
        lcfg = LoraConfig(
            r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
            task_type="CAUSAL_LM", target_modules=LORA_TARGETS
        )
        return get_peft_model(base, lcfg)

# ---- build base
base_model = build_base_model()

# ---- decide per TRAIN_MODE using only REPO_ID_IN as source of truth
in_files = _repo_files(REPO_ID_IN) if REPO_ID_IN else set()

if TRAIN_MODE == "FREEZE_BASE_ONLY_CLS":
    # no LoRA; freeze base; heads from REPO_ID_IN if present
    for p in base_model.parameters():
        p.requires_grad = False
    model = CausalLMClassifierModel(base_model, SIZES, LABEL_COLS, loss_type=LOSS_TYPE)
    if REPO_ID_IN and _repo_has_heads(in_files):
        _load_heads_from_repo_into(model, REPO_ID_IN, device="cpu")
    else:
        print("ℹ️ No heads found in REPO_ID_IN; starting heads from scratch.")

elif TRAIN_MODE == "LORA_TRAIN_BASE_FREEZE_CLS":
    # LoRA train only; heads from REPO_ID_IN and then freeze
    base_model = _maybe_attach_lora(base_model, in_files)
    model = CausalLMClassifierModel(base_model, SIZES, LABEL_COLS, loss_type=LOSS_TYPE)
    assert REPO_ID_IN and _repo_has_heads(in_files), \
        "For LORA_TRAIN_BASE_FREEZE_CLS you must provide trained heads in REPO_ID_IN."
    _load_heads_from_repo_into(model, REPO_ID_IN, device="cpu")
    for h in model.heads:
        for p in h.parameters():
            p.requires_grad = False
    print("✅ Heads frozen; training LoRA only.")

elif TRAIN_MODE == "LORA_AND_CLS":
    # joint train: LoRA + heads; init both from REPO_ID_IN when available
    base_model = _maybe_attach_lora(base_model, in_files)
    model = CausalLMClassifierModel(base_model, SIZES, LABEL_COLS, loss_type=LOSS_TYPE)
    if REPO_ID_IN and _repo_has_heads(in_files):
        _load_heads_from_repo_into(model, REPO_ID_IN, device="cpu")
        print("✅ Joint init: adapter (if present) + heads loaded from REPO_ID_IN.")
    else:
        print("ℹ️ Joint init without preexisting heads; heads start from scratch.")

else:
    raise ValueError("TRAIN_MODE must be one of {FREEZE_BASE_ONLY_CLS, LORA_TRAIN_BASE_FREEZE_CLS, LORA_AND_CLS}")

trainables = sum(p.numel() for p in model.parameters() if p.requires_grad)
total      = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainables:,} / {total:,}")


In [14]:
# Trainer setup
from transformers import DataCollatorWithPadding
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)


def compute_metrics(eval_pred):
    logits_concat, labels = eval_pred
    probs = 1.0 / (1.0 + np.exp(-logits_concat))

    metrics = {}
    start = 0
    all_true, all_pred = [], []

    for col, c in zip(LABEL_COLS, SIZES):
        p_slice = probs[:, start:start + c]
        y_slice = labels[:, start:start + c]
        pred = p_slice.argmax(-1)
        y    = y_slice.argmax(-1)
        metrics[f"accuracy_{col}"]    = accuracy_score(y, pred)
        metrics[f"f1_weighted_{col}"] = f1_score(y, pred, average="weighted")
        metrics[f"f1_micro_{col}"]    = f1_score(y, pred, average="micro")
        all_true.append(y); all_pred.append(pred)
        start += c

    metrics["f1_weighted_avg"]    = np.mean([metrics[f"f1_weighted_{c}"] for c in LABEL_COLS])
    metrics["accuracy_macro_avg"] = np.mean([metrics[f"accuracy_{c}"] for c in LABEL_COLS])
    y_all = np.concatenate(all_true, axis=0); p_all = np.concatenate(all_pred, axis=0)
    metrics["f1_micro_overall"]   = f1_score(y_all, p_all, average="micro")
    return metrics


In [ ]:
from transformers import TrainingArguments, Trainer

fp16 = (compute_dtype == torch.float16)
bf16 = (compute_dtype == torch.bfloat16)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=max(1, BATCH_SIZE),
    gradient_accumulation_steps=GRAD_ACC_STEPS,
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=LOG_STEPS,
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro_overall",
    greater_is_better=True,
    bf16=bf16,
    fp16=fp16,
    optim="adamw_torch",
    report_to=[],
    save_safetensors=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=proc_train,
    eval_dataset=proc_val,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

print("Quick sanity eval (pre-train)…")
trainer.evaluate()

In [ ]:
# Train
train_result = trainer.train()
trainer.save_state()
print(train_result)

In [ ]:
# %% [SAVE & PUSH → REPO_ID_OUT] — robust HTTP uploads (no deprecated Repository)
from huggingface_hub import HfApi, hf_hub_download
from peft import PeftModel as _PeftModel
from pathlib import Path
import json, torch, os

api = HfApi()

def save_heads_local(model, out_dir):
    out_dir = Path(out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    torch.save({"heads": [h.state_dict() for h in model.heads]}, out_dir / "cls_heads.pt")
    with open(out_dir / "cls_config.json", "w") as f:
        json.dump({"sizes": model.sizes, "tasks": model.task_names, "loss_type": model.loss_type}, f)
    return out_dir / "cls_heads.pt", out_dir / "cls_config.json"

def push_heads_http(repo_id, heads_pt_path, heads_cfg_path, hf_token=None):
    # Ensure repo exists; OK if already exists
    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True, token=hf_token)

    # Upload files via HTTP; works across hub versions and avoids git CLI
    api.upload_file(
        path_or_fileobj=str(heads_pt_path),
        path_in_repo="cls_heads.pt",
        repo_id=repo_id,
        repo_type="model",
        token=hf_token,
    )
    api.upload_file(
        path_or_fileobj=str(heads_cfg_path),
        path_in_repo="cls_config.json",
        repo_id=repo_id,
        repo_type="model",
        token=hf_token,
    )
    print(f"✅ Heads pushed to {repo_id} (HTTP upload)")

def maybe_push_adapter(model, repo_id, hf_token=None):
    # Push PEFT adapter if present (modes 2 & 3). No-op for FREEZE_BASE_ONLY_CLS.
    if isinstance(model.base, _PeftModel):
        model.base.push_to_hub(repo_id, token=hf_token)
        print(f"✅ Adapter pushed to {repo_id}")
        return True
    else:
        print("ℹ️ No PEFT adapter found to push (this is expected in FREEZE_BASE_ONLY_CLS).")
        return False

# ---- Save heads locally
heads_dir = Path(OUTPUT_DIR) / "heads_http"
pt_path, cfg_path = save_heads_local(model, heads_dir)

# ---- Push: adapter (if any) + heads (always)
_ = maybe_push_adapter(model, REPO_ID_OUT, hf_token=HF_TOKEN)
push_heads_http(REPO_ID_OUT, pt_path, cfg_path, hf_token=HF_TOKEN)


In [ ]:
# %% [RELOAD FOR EVAL/INFERENCE FROM REPO_ID_OUT] — robust loader
from huggingface_hub import HfApi, hf_hub_download
from peft import PeftModel as _PeftModel
from transformers import TrainingArguments, Trainer

api = HfApi()

def repo_has(repo_id, filename, hf_token=None):
    try:
        files = set(api.list_repo_files(repo_id=repo_id, repo_type="model", token=hf_token))
        return filename in files
    except Exception:
        return False

def has_adapter(repo_id, hf_token=None):
    # Any of these implies there is a PEFT adapter in the repo
    adapter_markers = {"adapter_config.json", "adapter_model.bin", "adapter_model.safetensors"}
    try:
        files = set(api.list_repo_files(repo_id=repo_id, repo_type="model", token=hf_token))
        return any(f in files for f in adapter_markers)
    except Exception:
        return False

def load_heads_from_repo_into(model, repo_id, hf_token=None, device="cpu"):
    cfg_path = hf_hub_download(repo_id, "cls_config.json", token=hf_token)
    with open(cfg_path, "r") as f:
        cfg = json.load(f)
    assert cfg["sizes"] == model.sizes and cfg["tasks"] == model.task_names, \
        f"Heads config mismatch. Expected sizes={model.sizes}, tasks={model.task_names} but got {cfg}."
    state_path = hf_hub_download(repo_id, "cls_heads.pt", token=hf_token)
    state = torch.load(state_path, map_location=device)
    for h, s in zip(model.heads, state["heads"]):
        h.load_state_dict(s)
    print(f"✅ Heads loaded from {repo_id}")

print("\n=== Reloading from REPO_ID_OUT for test eval ===")
# 1) Rebuild base
_reloaded_base = build_base_model()

# 2) If adapter exists in the repo, load it; otherwise use plain base
if has_adapter(REPO_ID_OUT, hf_token=HF_TOKEN):
    print(f"🔁 Loading adapter from {REPO_ID_OUT}")
    _reloaded_base = _PeftModel.from_pretrained(_reloaded_base, REPO_ID_OUT, token=HF_TOKEN)
else:
    print("ℹ️ No adapter found in REPO_ID_OUT; proceeding without LoRA.")

# 3) Wrap with classifier heads and load them
model_test = CausalLMClassifierModel(_reloaded_base, SIZES, LABEL_COLS, loss_type=LOSS_TYPE).eval()

if repo_has(REPO_ID_OUT, "cls_heads.pt", hf_token=HF_TOKEN) and repo_has(REPO_ID_OUT, "cls_config.json", hf_token=HF_TOKEN):
    load_heads_from_repo_into(model_test, REPO_ID_OUT, hf_token=HF_TOKEN, device="cpu")
else:
    raise RuntimeError("Expected classifier heads in REPO_ID_OUT but none found.")

# 4) Evaluate
_test_args = TrainingArguments(
    output_dir=str(Path(OUTPUT_DIR) / "_eval_tmp"),
    per_device_eval_batch_size=max(1, BATCH_SIZE),
)
_test_trainer = Trainer(
    model=model_test,
    args=_test_args,
    eval_dataset=proc_test,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
)

metrics = _test_trainer.evaluate(eval_dataset=proc_test)
print("\n📊 Test Metrics:")
for k, v in metrics.items():
    try:
        print(f"{k}: {float(v):.4f}")
    except Exception:
        print(k, ":", v)
